### Loading training, validation, and testing datasets

In [0]:
train = spark.read.table("revenue_operations.gold.delivery_risk_train")
val = spark.read.table("revenue_operations.gold.delivery_risk_val")
test = spark.read.table("revenue_operations.gold.delivery_risk_test")

#### Checking for class balance in training data

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

train.groupBy("late_delivery_flag_indexed").count().withColumn("total", F.sum("count").over(Window.partitionBy())).withColumn("percentage", F.col("count")/F.col("total")).display()

#### Dummy Baseline Model

In [0]:
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.sql import functions as F

# Find the majority class in training set
majority_class = train.groupBy("late_delivery_flag_indexed")\
    .count()\
        .orderBy("count", ascending = False)\
            .first()["late_delivery_flag_indexed"]

# Generate Baseline Prediction on validation set
baseline_prediction = val.withColumn("prediction", F.lit(majority_class))\
    .withColumn("label", F.col("late_delivery_flag_indexed"))


In [0]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Collect predictions to driver (convert to pandas)
preds_df = baseline_prediction.select("label", "prediction").toPandas()
y_true = preds_df["label"]
y_pred = preds_df["prediction"]

# Calculate all metrics using sklearn
accuracy = accuracy_score(y_true, y_pred)
precision_late = precision_score(y_true, y_pred, pos_label=1.0)
recall_late = recall_score(y_true, y_pred, pos_label=1.0)

# Display results
print(f"Baseline Model Performance:")
print(f"="*40)
print(f"Accuracy: {accuracy:.4f}")
print(f"Late-class Precision: {precision_late:.4f}")
print(f"Late-class Recall: {recall_late:.4f}")
print(f"\n" + "="*40)
print(f"\nFull Classification Report:")
print(classification_report(y_true, y_pred, target_names=["On-time", "Late"]))

# Display confusion matrix using sklearn's built-in visualization
print(f"\nConfusion Matrix:")
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, 
    display_labels=["On-time", "Late"],
    cmap="Blues"
)
plt.title("Baseline Model Confusion Matrix")
plt.show()

### MLFlow Experiment Setup

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Mlflow setup dynamically
current_user = spark.sql("SELECT current_user()").collect()[0][0]
experiment_name = f"/Users/{current_user}/revenue_operations/delivery_risk"
mlflow.set_experiment(experiment_name)

print(f"Using experiment : {experiment_name}")

# Helper function to expand Spark ML vector columns into individual binary columns
from pyspark.ml.functions import vector_to_array

def expand_vectors(df):
    """Expand Spark ML vector columns into separate binary columns"""
    # Expand customer_state_encoded vector
    df = df.withColumn("customer_state_array", vector_to_array("customer_state_encoded"))
    # Expand primary_product_category_encoded vector  
    df = df.withColumn("product_category_array", vector_to_array("primary_product_category_encoded"))
    
    # Drop original vector columns
    df = df.drop("customer_state_encoded", "primary_product_category_encoded")
    return df

print("Expanding encoded vector columns...")
train_expanded = expand_vectors(train)
val_expanded = expand_vectors(val)
print("✓ Vector columns expanded")

# Prepare data for sklearn by dropping non-numeric columns
# Exclude: order_id (ID), late_delivery_flag_indexed (target), timestamps
exclude_cols = [
    "order_id", 
    "late_delivery_flag_indexed",
    "order_approved_at", 
    "earliest_shipping_limit_date"
]
feature_cols = [col for col in train_expanded.columns if col not in exclude_cols]

print("\nConverting to pandas...")
# Converting Spark Dataframes to pandas
X_train = train_expanded.select(feature_cols).toPandas()
y_train = train_expanded.select("late_delivery_flag_indexed").toPandas().values.ravel()

X_val = val_expanded.select(feature_cols).toPandas()
y_val = val_expanded.select("late_delivery_flag_indexed").toPandas().values.ravel()

# The array columns need to be expanded into individual columns
import pandas as pd
import numpy as np

# Expand customer_state_array into separate columns
customer_state_df = pd.DataFrame(
    X_train['customer_state_array'].tolist(),
    columns=[f'customer_state_{i}' for i in range(len(X_train['customer_state_array'].iloc[0]))]
)

# Expand product_category_array into separate columns
product_category_df = pd.DataFrame(
    X_train['product_category_array'].tolist(),
    columns=[f'product_category_{i}' for i in range(len(X_train['product_category_array'].iloc[0]))]
)

# Drop array columns and concatenate expanded columns
X_train = X_train.drop(['customer_state_array', 'product_category_array'], axis=1)
X_train = pd.concat([X_train, customer_state_df, product_category_df], axis=1)

# Do the same for validation set
customer_state_df_val = pd.DataFrame(
    X_val['customer_state_array'].tolist(),
    columns=[f'customer_state_{i}' for i in range(len(X_val['customer_state_array'].iloc[0]))]
)

product_category_df_val = pd.DataFrame(
    X_val['product_category_array'].tolist(),
    columns=[f'product_category_{i}' for i in range(len(X_val['product_category_array'].iloc[0]))]
)

X_val = X_val.drop(['customer_state_array', 'product_category_array'], axis=1)
X_val = pd.concat([X_val, customer_state_df_val, product_category_df_val], axis=1)

print(f"✓ Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"✓ Validation set: {X_val.shape[0]} samples, {X_val.shape[1]} features")
print(f"  (includes {customer_state_df.shape[1]} customer state features + {product_category_df.shape[1]} product category features)")

# Convert any Decimal columns to float (required for MLflow serialization)
from decimal import Decimal
for col in X_train.columns:
    if X_train[col].dtype == 'object' and isinstance(X_train[col].iloc[0], Decimal):
        X_train[col] = X_train[col].astype(float)
        X_val[col] = X_val[col].astype(float)
print("✓ Converted Decimal columns to float")

# Scaling the X_train and X_val
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

#### Logistic Regression

In [0]:
with mlflow.start_run(run_name = "logistic_regression_baseline"):
    # Log Parameters to mlflow
    params = {
        "model_type" : "logistic_regression",
        "C" : 1.0,
        "max_iter" : 1000,
        "solver" : "lbfgs",
        "class_weight" : "balanced"
    }
    mlflow.log_params(params)

    # Train model
    print("\nTraining Logistic Regression..")
    model = LogisticRegression(
        C = params["C"],
        max_iter = params["max_iter"],
        solver = params["solver"],
        class_weight = params["class_weight"]
    )
    model.fit(X_train_scaled, y_train)
    print("Training complete")

    # Make prediction on validation set
    y_train_pred = model.predict(X_train_scaled)
    y_pred = model.predict(X_val_scaled)
    y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]

    # Calculate training metrics
    train_accuracy = accuracy_score(y_train, y_train_pred)
    train_precision = precision_score(y_train, y_train_pred, pos_label = 1.0)
    train_recall = recall_score(y_train, y_train_pred, pos_label = 1.0)

    # Calculate validation metrics
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, pos_label = 1.0)
    recall = recall_score(y_val, y_pred, pos_label = 1.0)
    f1 = f1_score(y_val, y_pred, pos_label = 1.0)

    # Log to Mlflow with "train_" prefix
    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("train_precision_late", train_precision)
    mlflow.log_metric("train_recall_late", train_recall)

    # Log to Mlflow with "test_" prefix
    mlflow.log_metric("val_accuracy", accuracy)
    mlflow.log_metric("val_precision_late", precision)
    mlflow.log_metric("val_recall_late", recall)

    # Log model
    mlflow.sklearn.log_model(
        model, "model",
        input_example = X_train[:5]  # Use original pandas DataFrame, not scaled numpy array
    )

    # Log confusion matrix as artifact
    fig, ax = plt.subplots(figsize = (8,6))
    ConfusionMatrixDisplay.from_predictions(
        y_val, y_pred,
        display_labels = ["On-time", "Late"],
        cmap = "Blues",
        ax = ax
    )
    plt.title("Logistic Regression Confusion Matrix")
    mlflow.log_figure(fig, "confison_matrix_log_reg.png")
    plt.close()

    # Get run info
    active_run = mlflow.active_run()
    run_id = active_run.info.run_id if active_run else "unknown"

    # Display results
    print("\n" + "="*40)
    print("Logistic Regression Baseline Results")
    print("="*40)
    print(f"Run ID: {run_id}")
    print(f"\nMetrics:")
    print(f"  Accuracy:           {accuracy:.4f}")
    print(f"  Precision (Late):   {precision:.4f}")
    print(f"  Recall (Late):      {recall:.4f}")
    print(f"  F1 Score (Late):    {f1:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_val, y_pred, target_names=["On-time", "Late"]))
    print("="*50)

    # Display confusion matrix
    ConfusionMatrixDisplay.from_predictions(
        y_val, y_pred,
        display_labels = ["On-time", "Late"],
        cmap = "Blues"
    )
    plt.title("Logistic Regression Confusion Matrix")
    plt.show()

print(f"\n Results saved to Mlflow experiment: {experiment_name}")
print(f"View in Mlflow UI")
